# 🦁 Model 4: MobileNetV2 Transfer Learning — Animal Image Classifier

**Author:** Samiun Mahmud | Member 2 — Baseline CNN & Edge Vision Lead  
**Branch:** `Samiun-Mahmud` | **Week 2**  
**Dataset:** 5-Class Animals (Cat, Cow, Dog, Lamb, Zebra) — 613 images  

---

## 🎯 Objective
Train **MobileNetV2** (pretrained on ImageNet, 1.4M params) as a lightweight transfer learning classifier for 5 animal classes — and compare it against the **Custom CNN** baseline (Model 3, 3.7M params) trained in Week 1.

## 📋 Two-Phase Transfer Learning Strategy

| Phase | What Trains | Learning Rate | Purpose |
|-------|------------|---------------|---------|
| **Phase 1** | Classification Head only | `1e-3` | Learn task-specific features using frozen ImageNet backbone |
| **Phase 2** | Head + Top 30 backbone layers | `1e-5` | Fine-tune deep features for animal-specific patterns |

---

## 🔧 Architecture Decisions
- **Input:** 224×224×3 (MobileNetV2 standard ImageNet resolution)
- **Preprocessing:** `mobilenet_v2.preprocess_input` → scales pixels to `[-1.0, 1.0]`
- **Backbone:** MobileNetV2 (`include_top=False`, `weights='imagenet'`)
- **Head:** GlobalAveragePooling2D → Dropout(0.3) → Dense(5, softmax)
- **Phase 1 LR:** 1e-3 (Adam) | **Phase 2 LR:** 1e-5 (AdamW fine-tuning)

## 📦 Section 1: Environment Setup & Dependencies

In [ ]:
# Verify GPU is available — always run on T4/V100 for speed
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

import numpy as np
import matplotlib.pyplot as plt
import os
print("✅ All imports successful")

In [ ]:
# Clone the project repository and switch to the Samiun-Mahmud branch
!git clone https://github.com/mohandevroy/Animal-Image-Classifier-.git
%cd Animal-Image-Classifier-
!git checkout Samiun-Mahmud
!git log --oneline -5

## 📂 Section 2: Dataset Loading

In [ ]:
# Mount Google Drive where the dataset folder is stored
from google.colab import drive
drive.mount('/content/drive')

# Update this path to match your Google Drive dataset folder
DATASET_PATH = '/content/drive/MyDrive/Dataset'  # <-- Change if needed
print(f"Dataset path: {DATASET_PATH}")

# Verify class subdirectories exist
classes = sorted(os.listdir(DATASET_PATH))
print(f"Detected classes: {classes}")
print(f"Number of classes: {len(classes)}")

In [ ]:
# Count images per class — check for class imbalance
print("Image count per class:")
total = 0
for cls in classes:
    cls_path = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        total += count
        print(f"  {cls:>10}: {count} images")
print(f"{'Total':>12}: {total} images")

In [ ]:
import sys
sys.path.insert(0, 'src')
from data_loader import load_and_split_data, get_default_image_size

# MobileNetV2 uses 224x224 and mobilenet_v2.preprocess_input automatically
print(f"MobileNetV2 default resolution: {get_default_image_size('mobilenet')}")

BATCH_SIZE = 32

train_ds, val_ds, test_ds, class_names = load_and_split_data(
    data_dir=DATASET_PATH,
    batch_size=BATCH_SIZE,
    model_type='mobilenet'   # auto-selects 224x224 + preprocess_input
)

print(f"\nClass names: {class_names}")
print(f"Train batches: {len(train_ds)}")
print(f"Val batches:   {len(val_ds)}")
print(f"Test batches:  {len(test_ds)}")

In [ ]:
# Visualise a sample batch — verify 224x224 resolution and preprocessing
sample_images, sample_labels = next(iter(train_ds))

# MobileNetV2 preprocess_input scales to [-1, 1] — unnormalize for display
display_imgs = (sample_images.numpy() + 1.0) / 2.0  # Back to [0, 1] for imshow
display_imgs = np.clip(display_imgs, 0, 1)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    if i < len(display_imgs):
        ax.imshow(display_imgs[i])
        ax.set_title(f"{class_names[sample_labels[i]]}\n(label={sample_labels[i].numpy()})",
                     fontsize=9)
        ax.axis('off')
plt.suptitle("Sample Training Images — MobileNetV2 Pipeline (224×224)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/mobilenet_sample_batch.png', dpi=150)
plt.show()
print(f"\nImage tensor shape: {sample_images.shape}")
print(f"Pixel range: [{sample_images.numpy().min():.3f}, {sample_images.numpy().max():.3f}]  ← Expected [-1.0, 1.0]")

## 🏗️ Section 3: MobileNetV2 Model Architecture

In [ ]:
from mobilenet import build_mobilenet_model

# Phase 1: Build with frozen backbone
model = build_mobilenet_model(
    input_shape=(224, 224, 3),
    num_classes=len(class_names),
    freeze_backbone=True,
    learning_rate=1e-3,
    dropout_rate=0.3
)

model.summary()

In [ ]:
# Count trainable vs frozen parameters
total_params = model.count_params()
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
frozen_params = total_params - trainable_params

print(f"{'Total parameters':<30}: {total_params:,}")
print(f"{'Trainable parameters':<30}: {trainable_params:,}  ← Classification head only")
print(f"{'Frozen parameters':<30}: {frozen_params:,}  ← ImageNet pretrained weights")
print(f"\nFrozen ratio: {frozen_params/total_params*100:.1f}% of parameters frozen in Phase 1")

## 🚂 Section 4: Phase 1 — Train Classification Head (Frozen Backbone)

In [ ]:
import os
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('logs/mobilenet/phase1', exist_ok=True)

PHASE1_EPOCHS = 15

phase1_callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir='logs/mobilenet/phase1', histogram_freq=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='models/mobilenetv2_best_phase1.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
]

print(f"Starting Phase 1 — {PHASE1_EPOCHS} epochs (backbone frozen)")
phase1_history = model.fit(
    train_ds,
    epochs=PHASE1_EPOCHS,
    validation_data=val_ds,
    callbacks=phase1_callbacks
)

In [ ]:
# Phase 1 results summary
best_p1_val_acc = max(phase1_history.history['val_accuracy'])
final_p1_val_acc = phase1_history.history['val_accuracy'][-1]
print(f"Phase 1 — Best val_accuracy : {best_p1_val_acc:.4f} ({best_p1_val_acc*100:.2f}%)")
print(f"Phase 1 — Final val_accuracy: {final_p1_val_acc:.4f} ({final_p1_val_acc*100:.2f}%)")

## 🔓 Section 5: Phase 2 — Fine-Tuning Top 30 Backbone Layers

In [ ]:
from mobilenet import unfreeze_mobilenet_top_layers

# Unfreeze top 30 layers of the MobileNetV2 backbone
# 10x lower learning rate to avoid catastrophic forgetting of ImageNet features
model = unfreeze_mobilenet_top_layers(model, num_layers=30, learning_rate=1e-5)

trainable_after = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"Trainable parameters after unfreezing: {trainable_after:,}")
print(f"Phase 2 learning rate: 1e-5 (10× lower than Phase 1)")

In [ ]:
os.makedirs('logs/mobilenet/phase2', exist_ok=True)

PHASE2_EPOCHS = 10

phase2_callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir='logs/mobilenet/phase2', histogram_freq=1),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='models/mobilenetv2_animal_classifier.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
]

print(f"Starting Phase 2 fine-tuning — {PHASE2_EPOCHS} epochs (top 30 layers unfrozen)")
phase2_history = model.fit(
    train_ds,
    epochs=PHASE2_EPOCHS,
    validation_data=val_ds,
    callbacks=phase2_callbacks
)

## 📊 Section 6: Learning Curves — Phase 1 + Phase 2 Combined

In [ ]:
# Merge Phase 1 and Phase 2 histories for combined visualisation
all_loss     = phase1_history.history['loss']     + phase2_history.history['loss']
all_val_loss = phase1_history.history['val_loss'] + phase2_history.history['val_loss']
all_acc      = phase1_history.history['accuracy']     + phase2_history.history['accuracy']
all_val_acc  = phase1_history.history['val_accuracy'] + phase2_history.history['val_accuracy']

total_epochs = len(all_loss)
phase1_len   = len(phase1_history.history['loss'])
epochs_range = range(1, total_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("MobileNetV2 (Model 4) — Combined Training History", fontsize=13, fontweight='bold')

# Loss plot
axes[0].plot(epochs_range, all_loss,     label='Train Loss',  color='teal',   linewidth=2)
axes[0].plot(epochs_range, all_val_loss, label='Val Loss',    color='orange', linewidth=2)
axes[0].axvline(phase1_len, color='gray', linestyle='--', linewidth=1.5, label=f'Phase 2 Start (Ep {phase1_len+1})')
axes[0].set_title('Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy plot
axes[1].plot(epochs_range, all_acc,     label='Train Accuracy', color='steelblue', linewidth=2)
axes[1].plot(epochs_range, all_val_acc, label='Val Accuracy',   color='seagreen',  linewidth=2)
axes[1].axvline(phase1_len, color='gray', linestyle='--', linewidth=1.5, label=f'Phase 2 Start (Ep {phase1_len+1})')
axes[1].set_title('Accuracy Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/mobilenet_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/mobilenet_curves.png")

## 🧪 Section 7: Test Set Evaluation

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint
best_model = tf.keras.models.load_model('models/mobilenetv2_animal_classifier.keras')

y_true, y_pred = [], []
for X, y in test_ds:
    preds = best_model.predict(X, verbose=0)
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("=" * 55)
print("     MOBILENETV2 — CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Confusion Matrix heatmap
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('MobileNetV2 — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/mobilenet_confusion_matrix.png', dpi=150)
plt.show()
print("Saved → results/mobilenet_confusion_matrix.png")

## ⚖️ Section 8: CNN vs MobileNetV2 — Head-to-Head Comparison

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# MobileNetV2 metrics just computed above
mobilenet_acc = accuracy_score(y_true, y_pred)
mobilenet_f1  = f1_score(y_true, y_pred, average='macro')

# Custom CNN results from Week 1 (actual training results from cnncl.ipynb)
cnn_test_acc = 1.0000   # 100% test accuracy (37 test images)
cnn_macro_f1 = 1.0000   # Perfect F1 across all 5 classes
cnn_params   = 3_697_653

# MobileNetV2 — total params from model summary
mobilenet_total_params = model.count_params()

comparison_data = {
    'Model'            : ['Custom CNN (Model 3)', 'MobileNetV2 (Model 4)'],
    'Input Resolution' : ['256×256', '224×224'],
    'Parameters'       : [f'{cnn_params:,}', f'{mobilenet_total_params:,}'],
    'Training Strategy': ['From scratch', 'Transfer Learning (ImageNet)'],
    'Test Accuracy'    : [f'{cnn_test_acc:.4f}', f'{mobilenet_acc:.4f}'],
    'Macro F1-Score'   : [f'{cnn_macro_f1:.4f}', f'{mobilenet_f1:.4f}'],
    'Preprocessing'    : ['[0, 1] (÷255)', '[-1, 1] (mobilenet_v2.preprocess_input)'],
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

# Save to CSV for Week 4 master benchmark table
df.to_csv('results/model_comparison.csv', index=False)
print("\nSaved → results/model_comparison.csv")

## 💾 Section 9: Save Results to GitHub Branch

In [ ]:
# Commit all result files to the Samiun-Mahmud branch
!git config user.email "samiunmahmud001@gmail.com"
!git config user.name "samiunmahmud23"
!git add results/mobilenet_curves.png results/mobilenet_confusion_matrix.png results/model_comparison.csv
!git status
!git commit -m "results(mobilenet): add MobileNetV2 training curves, confusion matrix and CNN vs MobileNetV2 benchmark CSV"
!git push origin Samiun-Mahmud

## ✅ Summary

| Item | Status |
|------|--------|
| MobileNetV2 Phase 1 (frozen backbone) | ✅ Complete |
| MobileNetV2 Phase 2 (fine-tuned top 30 layers) | ✅ Complete |
| Learning curves (Phase 1+2 combined) | ✅ Saved → `results/mobilenet_curves.png` |
| Confusion matrix | ✅ Saved → `results/mobilenet_confusion_matrix.png` |
| CNN vs MobileNetV2 comparison table | ✅ Saved → `results/model_comparison.csv` |
| Best model checkpoint | ✅ Saved → `models/mobilenetv2_animal_classifier.keras` |

**Next Step (Day 4):** Generate side-by-side comparison plots and push final benchmark CSV to the branch.